# **Getting Started with LlamaIndex**

In [ ]:
!pip install llama-index llama-index-llms-groq tavily-python -q -q

In [ ]:
import os
from google.colab import userdata
from llama_index.llms.groq import Groq
from tavily import AsyncTavilyClient
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

In [ ]:
# Store your Groq key in Colab secrets (key icon on left sidebar) as GROQ_API_KEY
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [ ]:
# With our API key in hand, and LlamaIndex installed, instantiating an LLM is one line:
llm = Groq(model="openai/gpt-oss-120b", api_key=os.environ["GROQ_API_KEY"])

# We can ask our LLM questions directly without an agent, just to check it's working:
response = llm.complete("What is the capital of South Dakota?")
print(response.text)

# **Creating an Agent**

A core feature of agents is that they can use tools to find out more about the state of the world, or take action in response to instructions, without us needing to be explicit about how and when they do that.

Tools extend what the LLM can accomplish in the real world. As an example, let's craft a helper function so the agent can reach beyond pure text.

Tools in LlamaIndex are just regular Python functions, so they can do anything a regular function can.

When creating a tool, its very important to:
- give the tool a distinctive name, and a clear description using docstrings. The LLM uses the name and description to understand what the tool does.
- annotate the types. This helps the LLM understand the expected input and output types.
- use async when possible, since this will make the workflow more efficient.

### **Maintaining State Across Agent Runs**

Short‑term memory can already accomplish a variety of tasks, but remembering past conversations is key to building truly context-aware agents. We'll see how to pass context forward so your agent doesn’t greet us like a stranger each time.

By default, the `AgentWorkflow` is stateless between runs. This means that the agent will not have any memory of previous runs.

To maintain state, we need to keep track of the previous state. In LlamaIndex, Workflows have a `Context` that can be used to maintain state within and between runs. Since the `AgentWorkflow` is just a pre-built `Workflow`, we can also use it now.

To maintain state between runs, we'll create a new `Context` called `ctx`. We pass in our `workflow` to properly configure this Context object for the workflow that will use it.

With our configured Context, we can pass it to our first run.

### **Define Tool**

In [ ]:
# Note the type annotations for the incoming query and the return string
async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient()
    return str(await client.search(query))

### **Instantiating an AgentWorkflow**

In [ ]:
# stateful_workflow = AgentWorkflow.from_tools_or_functions(
#     [set_name, search_web],
#     llm=llm,
#     system_prompt="You are a helpful assistant that answers questions. If you don't know the answer, you can search the web for information.",
#     initial_state={"name": "Ashutosh"}
# )

In [ ]:
workflow = AgentWorkflow.from_tools_or_functions(
    [search_web],
    llm=llm,
    system_prompt="You are a helpful assistant that answers questions. If you don't know the answer, you can search the web for information.",
)


# Configure a context to work with our workflow
ctx = Context(workflow)

response = await workflow.run(
    user_msg="My name is Laurie, nice to meet you!", ctx=ctx
)
print(str(response))

Now we can pass the same context to a second run, and it will remember what happened before:

In [ ]:
# Run the workflow again with the same context
response = await workflow.run(user_msg="What is my name?", ctx=ctx)
print(str(response))

### 🌐 **Accessing Context from within tools**

Sometimes our tools need an outlook on the wider world. Let's see how a tool can inspect the running context so its output remains focused on the current task.

By defining our tool to have access to the workflow context, we can set and retrieve variables from the context and use them in the tool or between tools.

`AgentWorkflow` uses a context variable called `state` that gets passed to every agent. We can rely on information in `state` being available without explicitly having to pass it in.

_**Note:**_ To access the `Context`, the Context parameter should be the first parameter of the tool.

### **Streaming output and events**

When the expected answer is long and we want to immediately start returning output, we can use live token streams and event callbacks to see progress instantly.

🎞️ The `AgentWorkflow` also supports streaming. The `AgentWorkflow` can be streamed like any other `Workflow`. This works by using the handler that is returned from the workflow. The stream returns a variety of event types as the workflow executes, and we can select which ones to handle.

* If we want to stream the LLM output, we can use the `AgentStream` events, which contain a `delta` of the new output each time
* `AgentInput` events will tell us which agent is running (our current workflow just has one agent)
* `AgentOutput` events will tell us what the agents returned, including which tools they called
* `ToolCall` and `ToolCallResults` will track tools as they are called and their outputs

In this example we're handling the `AgentStream` events. We can tell our handling is working because the output will appear in chunks as we run the cell, rather than appearing all at once.

In [ ]:
from llama_index.core.workflow import Context

# We've set the first parameter to be of type Context
async def set_name(ctx: Context, name: str) -> str:
    state = await ctx.get("state")
    state["name"] = name
    await ctx.set("state", state)
    return f"Name set to {name}"


stateful_workflow = AgentWorkflow.from_tools_or_functions(
    [set_name, search_web],
    llm=llm,
    system_prompt="You are a helpful assistant that can set a name.",
    initial_state={"name": "Laurie"},
)

stateful_workflow_context = Context(stateful_workflow)

# Running the agent
handler = stateful_workflow.run(user_msg="What is the weather in Saskatoon?", ctx=stateful_workflow_context)

async for event in handler.stream_events():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

#### **Streaming output and events**

In [ ]:
handler = workflow.run(user_msg="What is the weather in Saskatoon?")
################################################### Logging ###########################################################
async for event in handler.stream_events():
    if isinstance(event, AgentInput):
       print("Agent input: ", event.input)  # Current input messages
       print("Agent name:", event.current_agent_name)  # Current agent name
    elif isinstance(event, AgentOutput):
       print("Agent output: ", event.response)  # Current full response
       print("Tool calls made: ", event.tool_calls)  # Selected tool calls, if any
       print("Raw LLM response: ", event.raw)  # Raw llm api response
    elif isinstance(event, ToolCallResult):
       print("Tool called: ", event.tool_name)  # Tool name
       print("Arguments to the tool: ", event.tool_kwargs)  # Tool kwargs
       print("Tool output: ", event.tool_output)  # Tool output

# **Multi-agent System with AgentWorkflow**


Our system will have three agents:

* 🔎 A `ResearchAgent` that searches the web for information on the given topic.
* 📝 A `WriteAgent` that will write the report using the information found by the `ResearchAgent`.
* 🧐 A `ReviewAgent` that reviews the report and provides feedback.

There are a lot of ways we could go about building a system to perform this task. In this example, we will use a few tools to help with the research and writing processes.

1. A `web_search` tool to search the web for information on the given topic.
2. A `record_notes` tool which will save research found on the web to the state so that the other tools can use it.
3. A `write_report` tool to write the report using the information found by the `ResearchAgent`.
4. A `review_report` tool to review the report and provide feedback.

Utilizing the `Context` class, we can pass state between agents, and each agent will have access to the current state of the system.

In [ ]:
import os
from google.colab import userdata
from llama_index.llms.groq import Groq
from tavily import AsyncTavilyClient
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)
from llama_index.core.workflow import Context

### **Create Tools**

In [ ]:
async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient()
    return str(await client.search(query))


async def record_notes(ctx: Context, notes: str, notes_title: str = "Untitled Notes") -> str:
    """Useful for recording notes on a given topic."""
    async with ctx.store.edit_state() as state:
        if "research_notes" not in state:
            state["research_notes"] = {}
        state["research_notes"][notes_title] = notes
    return "Notes recorded."


async def write_report(ctx: Context, report_content: str) -> str:
    """Useful for writing a report on a given topic."""
    async with ctx.store.edit_state() as state:
        state["report_content"] = report_content
    return "Report written."


async def review_report(ctx: Context, review: str) -> str:
    """Useful for reviewing a report and providing feedback."""
    async with ctx.store.edit_state() as state:
        state["review"] = review
    return "Report reviewed."

### **Create Agents**
With our tools defined, we can now create our agents.

If the LLM you are using supports tool calling (most big models do), you can use the `FunctionAgent` class, which is a little more efficient. Otherwise, you can use the `ReActAgent` class.

Here, the name and description of each agent is used so that the system knows what each agent is responsible for and when to hand off control to the next agent.

Each agent takes a `system_prompt` that tells it what it should do, and suggests how to work with the other agents.

You can also optionally help your multi-agent system constrain itself by listing which other agents an agent can talk to using `can_handoff_to` (otherwise it will just try to figure this out on its own).

In [ ]:
research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Useful for searching the web for information "
    "on a given topic and recording notes on the topic.",
    system_prompt=(
        "You are the ResearchAgent that can search the web for "
        "information on a given topic and record notes on the topic. "
        "Once notes are recorded, you should hand off "
        "control to the WriteAgent to write a report on the topic."
    ),
    llm=llm,
    tools=[search_web, record_notes],
    can_handoff_to=["WriteAgent"],
)

write_agent = FunctionAgent(
    name="WriteAgent",
    description="Useful for writing a report on a given topic.",
    system_prompt=(
        "You are the WriteAgent that can write a report on a given topic. "
        "Your report should be in a markdown format. The content should be grounded in the research notes. "
        "Once the report is written, you should get feedback from the ReviewAgent."
    ),
    llm=llm,
    tools=[write_report],
    can_handoff_to=["ReviewAgent", "ResearchAgent"],
)

review_agent = FunctionAgent(
    name="ReviewAgent",
    description="Useful for reviewing a report and providing feedback.",
    system_prompt=(
        "You are the ReviewAgent that can review a report and provide feedback. "
        "Your feedback should either approve the current report or request changes for the WriteAgent to implement."
    ),
    llm=llm,
    tools=[review_report],
)

### **Running the Workflow**

**Action!** With roles assigned, watch the ensemble solve tasks in concert. We'll gain insight into cross‑agent collaboration and how the workflow coordinates their moves.

After having defined our agents, we can create our `AgentWorkflow` and run it. Since this is a multi-agent system, instantiating it looks a little different. Instead of using the `from_tools_or_functions` helper, we instantiate it directly. It takes these arguments:
* an array of agents
* the name of the agent that it should start with (this will receive the initial `user_msg`)
* an `initial_state` which populates the context variable `state` we mentioned earlier

In [ ]:
agent_workflow = AgentWorkflow(
    agents=[research_agent, write_agent, review_agent],
    root_agent=research_agent.name,
    initial_state={
        "research_notes": {},
        "report_content": "Not written yet.",
        "review": "Review required.",
    },
)

#### Create an explicit, reusable context

In [ ]:
# agent_workflow_context = Context(agent_workflow)

### **Run the workflow, streaming progress, using the explicit context**

In [ ]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

handler = agent_workflow.run(
    user_msg="""Write me a report on the history of the web.
      Briefly describe the history of the world wide web, including
      the development of the internet and the development of the web,
      including 21st century developments""",
    max_handoffs=1
)

current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")

In [ ]:
# After your streaming loop finishes (or instead of it, if you just want the end result)
final_state = await handler.ctx.store.get_state()

print("\n" + "="*50)
print("📄 FINAL REPORT")
print("="*50 + "\n")
print(final_state["report_content"])

In [ ]:
# final_state = await agent_workflow_context.store.get_state()

# print("="*50)
# print("FINAL REPORT")
# print("="*50)
# print(final_state["report_content"])

# print("\n" + "="*50)
# print("REVIEW FEEDBACK")
# print("="*50)
# print(final_state["review"])

In [ ]:
# handler = agent_workflow.run(
#     user_msg="""Write me a report on the history of the web.
#       Briefly describe the history of the world wide web, including
#       the development of the internet and the development of the web,
#       including 21st century developments""",
#     ctx=agent_workflow_context,
#     max_handoffs=5,   # bumped from 1 -- a single handoff can't reach Research -> Write -> Review
# )

# current_agent = None
# async for event in handler.stream_events():
#     if (
#         hasattr(event, "current_agent_name")
#         and event.current_agent_name != current_agent
#     ):
#         current_agent = event.current_agent_name
#         print(f"\n{'='*50}")
#         print(f"🤖 Agent: {current_agent}")
#         print(f"{'='*50}\n")
#     elif isinstance(event, AgentOutput):
#         if event.response.content:
#             print("📤 Output:", event.response.content)
#         if event.tool_calls:
#             print(
#                 "🛠️  Planning to use tools:",
#                 [call.tool_name for call in event.tool_calls],
#             )
#     elif isinstance(event, ToolCallResult):
#         print(f"🔧 Tool Result ({event.tool_name}):")
#         print(f"  Arguments: {event.tool_kwargs}")
#         print(f"  Output: {event.tool_output}")
#     elif isinstance(event, ToolCall):
#         print(f"🔨 Calling Tool: {event.tool_name}")
#         print(f"  With arguments: {event.tool_kwargs}")

# # Resolve the handler to get the workflow's final return value
# final_response = await handler

### **Print the final report (pulled straight from state — the reliable source)**

In [ ]:
# final_state = await agent_workflow_context.store.get_state()

# print("="*50)
# print("FINAL REPORT")
# print("="*50)
# print(final_state["report_content"])

# print("\n" + "="*50)
# print("REVIEW FEEDBACK")
# print("="*50)
# print(final_state["review"])

# **Building agentic workflows from scratch**


`AgentWorkflow` is a pre-built `Workflow`

We can easily build our own workflows directly to create multi-agent systems that are customized to our needs. There's a number of reasons we might want to branch out from AgentWorkflow to rolling your own workflow:

* 🎛️ **Fine-grained flow control**: Workflows allow looping, branching, parallel calls and map-reduce behavior. Workflows allow  to precisely specify how data gets passed around system.

* 🔁 **Structured inputs and outputs**: If we want our inputs and outputs to be richer than simple strings, or to kick off with more than just a `user_msg`, workflows allow to do that.

* 🎨 **Multimodality**: LlamaIndex can handle more than just text! You we handle images, audio and video as well.

* 🧭 **Query planning**: a custom workflow can execute complex planning of how to run based on its inputs, rather than immediately jumping into a team.

* 🔍 **Reflection**: A powerful technique for agents is their ability to examine their own output and decide whether it's sufficient, or if they need to try again. Using looping, we can implement this pattern.


### **Creating a new workflow**

Under the hood, Workflows are regular Python classes. They are defined as a series of `steps`, each of which receives certain classes of events and emits certain classes of events.

Here's the most basic form of a workflow, with a single step:

In [ ]:
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
)

class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # Do something here
        return StopEvent(result="Hello, world!")

This new `MyWorkflow` class:
* Uses the `@step` decorator to declare a function to be a step
* Has a single step called `my_step` which accepts a `StartEvent`. `StartEvent` is a special event which is always generated when a workflow first runs.
* `my_step` returns a `StopEvent`, which is another special event. When a `StopEvent` is emitted the workflow returns it and stops running.

We instantiate it and run it just like we ran the `AgentWorkflow`:

In [ ]:
workflow = MyWorkflow(timeout=10, verbose=False)
result = await workflow.run()
print(result)

## **</> Aside: running a workflow in regular Python**

Workflows are async by default, so we use `await` to get the result of the `run` command. This will work fine in a notebook environment; if you are using a vanilla Python script, you will need to import `asyncio` and wrap your code in an async function, like this:

In [ ]:
# async def main():
#     w = MyWorkflow(timeout=10, verbose=False)
#     result = await w.run()
#     print(result)


# if __name__ == "__main__":
#     import asyncio
#     asyncio.run(main())

Since we're in a notebook right now, we won't execute the above code as it won't work!

### **Visualizing a workflow**

A great feature of workflows is the built-in visualizer, which we will install now:


In [ ]:
!pip install llama-index-utils-workflow -q -q

Let's visualize the simple workflow we just created:

In [ ]:
from llama_index.utils.workflow import draw_all_possible_flows

draw_all_possible_flows(MyWorkflow, filename="basic_workflow.html")

#  **Designing multi-step workflows with custom events**

### **Custom Events**

Multiple steps are created by defining custom events that can be emitted by steps and trigger other steps. Let's define a simple 3-step workflow by defining two custom events, `FirstEvent` and `SecondEvent`. These classes can have any names and properties, but must inherit from `Event`.


In [ ]:
from llama_index.core.workflow import (
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)

class FirstEvent(Event):
    first_output: str

class SecondEvent(Event):
    second_output: str

### **Defining the workflow**

Now we define the workflow itself. We do this by defining the input and output types on each step.

* `step_one` takes a `StartEvent` and returns a `FirstEvent`
* `step_two` takes a `FirstEvent` and returns a `SecondEvent`
* `step_three` takes a `SecondEvent` and returns a `StopEvent`

In [ ]:
class MyWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> FirstEvent:
        print(ev.first_input)
        return FirstEvent(first_output="First step complete.")

    @step
    async def step_two(self, ev: FirstEvent) -> SecondEvent:
        print(ev.first_output)
        return SecondEvent(second_output="Second step complete.")

    @step
    async def step_three(self, ev: SecondEvent) -> StopEvent:
        print(ev.second_output)
        return StopEvent(result="Workflow complete.")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run(first_input="Start the workflow.")
print(result)

### **Looping**

However, there's not much point to a workflow if it just runs straight through! A key feature of Workflows is their enablement of branching and looping logic, more simply and flexibly than graph-based approaches. To enable looping, we'll create a new `LoopEvent` (LoopEvent is not special, any event can be used to loop).

In [ ]:
class LoopEvent(Event):
    first_input: str

In [ ]:
import random

class MyWorkflow(Workflow):

    # Step one will trigger on a StartEvent or a LoopEvent
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> FirstEvent:
        print(ev.first_input)
        return FirstEvent(first_output="First step complete")

    # Step two returns either a SecondEvent or a LoopEvent
    @step
    async def step_two(self, ev: FirstEvent) -> SecondEvent | LoopEvent:
        print(ev.first_output)
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(first_input="Back to step one.")
        else:
            print("Good thing happened")
            return SecondEvent(second_output="Second step complete.")

    @step
    async def step_three(self, ev: SecondEvent) -> StopEvent:
        print(ev.second_output)
        return StopEvent(result="Workflow complete.")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run(first_input="Start the workflow.")
print(result)

### **Branching**

The same constructs that allow us to loop allow us to create branches. Here's a workflow that executes two different branches depending on an early decision:

In [ ]:
class BranchA1Event(Event):
    payload: str


class BranchA2Event(Event):
    payload: str


class BranchB1Event(Event):
    payload: str


class BranchB2Event(Event):
    payload: str


class BranchWorkflow(Workflow):
    @step
    async def start(self, ev: StartEvent) -> BranchA1Event | BranchB1Event:
        if random.randint(0, 1) == 0:
            print("Go to branch A")
            return BranchA1Event(payload="Branch A")
        else:
            print("Go to branch B")
            return BranchB1Event(payload="Branch B")

    @step
    async def step_a1(self, ev: BranchA1Event) -> BranchA2Event:
        print(ev.payload)
        return BranchA2Event(payload=ev.payload)

    @step
    async def step_b1(self, ev: BranchB1Event) -> BranchB2Event:
        print(ev.payload)
        return BranchB2Event(payload=ev.payload)

    @step
    async def step_a2(self, ev: BranchA2Event) -> StopEvent:
        print(ev.payload)
        return StopEvent(result="Branch A complete.")

    @step
    async def step_b2(self, ev: BranchB2Event) -> StopEvent:
        print(ev.payload)
        return StopEvent(result="Branch B complete.")

# **Concurrency and event collection**
The final form of flow control we can implement in workflows is concurrent execution.

🤹 This allows us to efficiently run long-running tasks in parallel, and gather them together when they are needed. This can also let us perform map-reduce style tasks.

To do this, we'll be using the `Context` object that we already encountered when working with `AgentWorkflow`. The `Context` is available to every step in a workflow: to access it, declare it as an argument to our step and it will be automatically populated.

In this example, we use a new method, `Context.send_event` rather than returning an event. This allows us to emit multiple events in parallel rather than returning just one as we have previously.

In [ ]:
import asyncio
import random
from llama_index.core.workflow import (
    Context,
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)

class StepTwoEvent(Event):
    query: str

class ParallelFlow(Workflow):
    @step
    async def start(self, ctx: Context, ev: StartEvent) -> StepTwoEvent:
        ctx.send_event(StepTwoEvent(query="Query 1"))
        ctx.send_event(StepTwoEvent(query="Query 2"))
        ctx.send_event(StepTwoEvent(query="Query 3"))

    @step(num_workers=4)
    async def step_two(self, ctx: Context, ev: StepTwoEvent) -> StopEvent:
        print("Running slow query ", ev.query)
        await asyncio.sleep(random.randint(1, 5))

        return StopEvent(result=ev.query)

In [ ]:
w = ParallelFlow(timeout=10, verbose=False)
result = await w.run(message="Start the workflow.")
print(result)

As we can see, it executes all 3 queries. The first one to complete emits a StopEvent, at which point the workflow halts without waiting for the other 2 events.

### **Collecting events**
But what if we do want the output of all 3 events? Another method, `Context.collect_events`, exists for that purpose:

In [ ]:
class StepThreeEvent(Event):
    result: str

class ConcurrentFlow(Workflow):
    @step
    async def start(self, ctx: Context, ev: StartEvent) -> StepTwoEvent:
        ctx.send_event(StepTwoEvent(query="Query 1"))
        ctx.send_event(StepTwoEvent(query="Query 2"))
        ctx.send_event(StepTwoEvent(query="Query 3"))

    @step(num_workers=4)
    async def step_two(self, ctx: Context, ev: StepTwoEvent) -> StepThreeEvent:
        print("Running query ", ev.query)
        await asyncio.sleep(random.randint(1, 5))
        return StepThreeEvent(result=ev.query)

    @step
    async def step_three(self, ctx: Context, ev: StepThreeEvent) -> StopEvent:
        # Wait until we receive 3 events
        result = ctx.collect_events(ev, [StepThreeEvent] * 3)
        if result is None:
            print("Not all events received yet.")
            return None

        # Do something with all 3 results together
        print(result)
        return StopEvent(result="Done")

In [ ]:
w = ConcurrentFlow(timeout=10, verbose=False)
result = await w.run(message="Start the workflow.")
print(result)

In the above example, we emit three `StepTwoEvent`s which trigger `step_two`, each execution of which eventually emits a `StepThreeEvent`.

`step_three` gets triggered whenever a `StepThreeEvent` fires. What `collect_events` does is store the events in the context until it has collected the number and type of events specified in its second argument. In this case, we've told it to wait for 3 events.

If an event fires and `collect_events` hasn't yet seen the right number of events, it returns `None`, so we tell `step_three` to do nothing in that case. When `collect_events` receives the right number of events it returns them as an array, which you can see us printing in the final output. Note that in the array they are stored in the order they returned, not the order they were emitted.

To implement a map-reduce pattern, we would split our task up into as many steps as necessary, and use `Context` to store that number with `ctx.set("num_events", some_number)`. Then in `step_three` you would wait for the number stored in the context using `await ctx.get("num_events")`. So you don't need to know in advance exactly how many concurrent steps you're taking.

### **Collecting different event types**

We don't just have to wait for multiple events of the same kind. In this example, we'll emit 3 totally different events and collect them at the end.

In [ ]:
class StepAEvent(Event):
    query: str

class StepACompleteEvent(Event):
    result: str

class StepBEvent(Event):
    query: str

class StepBCompleteEvent(Event):
    result: str

class StepCEvent(Event):
    query: str

class StepCCompleteEvent(Event):
    result: str

In [ ]:
class ConcurrentFlow(Workflow):
    @step
    async def start(
        self, ctx: Context, ev: StartEvent
    ) -> StepAEvent | StepBEvent | StepCEvent:
        ctx.send_event(StepAEvent(query="Query 1"))
        ctx.send_event(StepBEvent(query="Query 2"))
        ctx.send_event(StepCEvent(query="Query 3"))

    @step
    async def step_a(self, ctx: Context, ev: StepAEvent) -> StepACompleteEvent:
        print("Doing something A-ish")
        return StepACompleteEvent(result=ev.query)

    @step
    async def step_b(self, ctx: Context, ev: StepBEvent) -> StepBCompleteEvent:
        print("Doing something B-ish")
        return StepBCompleteEvent(result=ev.query)

    @step
    async def step_c(self, ctx: Context, ev: StepCEvent) -> StepCCompleteEvent:
        print("Doing something C-ish")
        return StepCCompleteEvent(result=ev.query)

    @step
    async def step_three(
        self,
        ctx: Context,
        ev: StepACompleteEvent | StepBCompleteEvent | StepCCompleteEvent,
    ) -> StopEvent:
        print("Received event ", ev.result)

        # Wait until we receive 3 events
        events = ctx.collect_events(
            ev,
            [StepCCompleteEvent, StepACompleteEvent, StepBCompleteEvent],
        )
        if (events is None):
            return None

        # Do something with all 3 results together
        print("All events received: ", events)
        return StopEvent(result="Done")

When we run, it will do all three things and wait for them in `step_three`.

In [ ]:
w = ConcurrentFlow(timeout=10, verbose=False)
result = await w.run(message="Start the workflow.")
print(result)

Note that the order of the events is significant: `collect_events` was told to expect Step C, then A, then B, and that's the order they are in the return array.

# **Putting together a multi-agent system**

Time to combine everything we've learned into a robust, production‑ready solution that juggles context, tools, and multiple agents: our own multi-agent system!

Our Deep Research multi-agent system will have three agents:
* ❓ `QuestionAgent` which accepts a research topic and generates a bunch of questions
* 💬 `AnswerAgent` which answers a specific question (we'll need to call it many times)
* 📝 `ReportAgent` which aggregates all the answers and generates a report.

We'll create these as `FunctionAgent`s, the same as we did when creating agents for `AgentWorkflow`.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
# from llama_index.llms.openai import OpenAI
import os

question_agent = FunctionAgent(
    tools=[],
    llm=llm,
    verbose=False,
    system_prompt="""You are part of a deep research system.
      Given a research topic, you should come up with a bunch of questions
      that a separate agent will answer in order to write a comprehensive
      report on that topic. To make it easy to answer the questions separately,
      you should provide the questions one per line. Don't include markdown
      or any preamble in your response, just a list of questions."""
)
# Keep answer_agent as a FunctionAgent since it genuinely uses search_web
answer_agent = FunctionAgent(
    tools=[search_web],
    llm=llm,
    verbose=False,
    system_prompt="""You are part of a deep research system.
      Given a specific question, your job is to come up with a deep answer
      to that question, which will be combined with other answers on the topic
      into a comprehensive report. You can search the web to get information
      on the topic, as many times as you need."""
)
report_agent = FunctionAgent(
    tools=[],
    llm=llm,
    verbose=False,
    system_prompt="""You are part of a deep research system.
      Given a set of answers to a set of questions, your job is to combine
      them all into a comprehensive report on the topic."""
)

In [ ]:
# from llama_index.core.agent.workflow import FunctionAgent

# answer_agent = FunctionAgent(
#     tools=[search_web],
#     llm=llm,
#     verbose=False,
#     system_prompt="""You are part of a deep research system.
#       Given a specific question, your job is to come up with a deep answer
#       to that question, which will be combined with other answers on the topic
#       into a comprehensive report. You can search the web to get information
#       on the topic, as many times as you need."""
# )

The Workflow we'll need to handle this task needs to do a few things:
* Accept the topic and pass it to the QuestionAgent
* Take all the answers from the QuestionAgent and split them up, firing off one AnswerAgent for each question
* Aggregate all the questions and answers from the AnswerAgents
* Generate a single report from them

In [ ]:
from llama_index.core.workflow import Event, Context
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step
)

class GenerateEvent(Event):
    research_topic: str

class QuestionEvent(Event):
    question: str

class AnswerEvent(Event):
    question: str
    answer: str

class ProgressEvent(Event):
    msg: str

class DeepResearchWorkflow(Workflow):

    @step
    async def setup(self, ctx: Context, ev: StartEvent) -> GenerateEvent:
        self.question_agent = ev.question_agent
        self.answer_agent = ev.answer_agent
        self.report_agent = ev.report_agent

        ctx.write_event_to_stream(ProgressEvent(msg="Starting research"))

        return GenerateEvent(research_topic=ev.research_topic)

    @step
    async def generate_questions(self, ctx: Context, ev: GenerateEvent) -> QuestionEvent:

        # FIXED: ctx.set -> await ctx.store.set
        await ctx.store.set("research_topic", ev.research_topic)
        ctx.write_event_to_stream(ProgressEvent(msg=f"Research topic is {ev.research_topic}"))

        result = await self.question_agent.run(user_msg=f"""Generate some questions
          on the topic <topic>{ev.research_topic}</topic>.""")

        lines = str(result).split("\n")
        questions = [line.strip() for line in lines if line.strip() != ""]

        # FIXED: ctx.set -> await ctx.store.set
        await ctx.store.set("total_questions", len(questions))

        for question in questions:
            ctx.send_event(QuestionEvent(question=question))

    @step
    async def answer_question(self, ctx: Context, ev: QuestionEvent) -> AnswerEvent:

        result = await self.answer_agent.run(user_msg=f"""Research the answer to this
          question: <question>{ev.question}</question>. You can use web
          search to help you find information on the topic, as many times
          as you need. Return just the answer without preamble or markdown.""")

        ctx.write_event_to_stream(ProgressEvent(msg=f"""Received question {ev.question}
            Came up with answer: {str(result)}"""))

        return AnswerEvent(question=ev.question, answer=str(result))

    @step
    async def write_report(self, ctx: Context, ev: AnswerEvent) -> StopEvent:

        # FIXED: ctx.get -> await ctx.store.get
        research = ctx.collect_events(ev, [AnswerEvent] * await ctx.store.get("total_questions"))
        if research is None:
            ctx.write_event_to_stream(ProgressEvent(msg="Collecting answers..."))
            return None

        ctx.write_event_to_stream(ProgressEvent(msg="Generating report..."))

        all_answers = ""
        for q_and_a in research:
            all_answers += f"Question: {q_and_a.question}\nAnswer: {q_and_a.answer}\n\n"

        # FIXED: ctx.get -> await ctx.store.get
        result = await self.report_agent.run(user_msg=f"""You are part of a deep research system.
          You have been given a complex topic on which to write a report:
          <topic>{await ctx.store.get("research_topic")}.

          Other agents have already come up with a list of questions about the
          topic and answers to those questions. Your job is to write a clear,
          thorough report that combines all the information from those answers.

          Here are the questions and answers:
          <questions_and_answers>{all_answers}</questions_and_answers>""")

        return StopEvent(result=str(result))

In [ ]:
class DeepResearchWorkflow(Workflow):

    @step
    async def setup(self, ctx: Context, ev: StartEvent) -> GenerateEvent:
        self.llm = ev.llm  # store the raw llm
        self.answer_agent = ev.answer_agent

        ctx.write_event_to_stream(ProgressEvent(msg="Starting research"))

        return GenerateEvent(research_topic=ev.research_topic)

    @step
    async def generate_questions(self, ctx: Context, ev: GenerateEvent) -> QuestionEvent:

        await ctx.store.set("research_topic", ev.research_topic)
        ctx.write_event_to_stream(ProgressEvent(msg=f"Research topic is {ev.research_topic}"))

        prompt = f"""You are part of a deep research system.
          Given a research topic, you should come up with a bunch of questions
          that a separate agent will answer in order to write a comprehensive
          report on that topic. To make it easy to answer the questions separately,
          you should provide the questions one per line. Don't include markdown
          or any preamble in your response, just a list of questions.

          Topic: {ev.research_topic}"""

        # Call the LLM directly, bypassing FunctionAgent's tool_choice logic
        result = await self.llm.acomplete(prompt)

        lines = str(result).split("\n")
        questions = [line.strip() for line in lines if line.strip() != ""]

        await ctx.store.set("total_questions", len(questions))

        for question in questions:
            ctx.send_event(QuestionEvent(question=question))

    @step
    async def answer_question(self, ctx: Context, ev: QuestionEvent) -> AnswerEvent:

        result = await self.answer_agent.run(user_msg=f"""Research the answer to this
          question: <question>{ev.question}</question>. You can use web
          search to help you find information on the topic, as many times
          as you need. Return just the answer without preamble or markdown.""")

        ctx.write_event_to_stream(ProgressEvent(msg=f"""Received question {ev.question}
            Came up with answer: {str(result)}"""))

        return AnswerEvent(question=ev.question, answer=str(result))

    @step
    async def write_report(self, ctx: Context, ev: AnswerEvent) -> StopEvent:

        research = ctx.collect_events(ev, [AnswerEvent] * await ctx.store.get("total_questions"))
        if research is None:
            ctx.write_event_to_stream(ProgressEvent(msg="Collecting answers..."))
            return None

        ctx.write_event_to_stream(ProgressEvent(msg="Generating report..."))

        all_answers = ""
        for q_and_a in research:
            all_answers += f"Question: {q_and_a.question}\nAnswer: {q_and_a.answer}\n\n"

        prompt = f"""You are part of a deep research system.
          You have been given a complex topic on which to write a report:
          <topic>{await ctx.store.get("research_topic")}.

          Other agents have already come up with a list of questions about the
          topic and answers to those questions. Your job is to write a clear,
          thorough report that combines all the information from those answers.

          Here are the questions and answers:
          <questions_and_answers>{all_answers}</questions_and_answers>"""

        # Call the LLM directly, bypassing FunctionAgent's tool_choice logic
        result = await self.llm.acomplete(prompt)

        return StopEvent(result=str(result))

In [ ]:
import asyncio
from openai import RateLimitError

async def call_with_retry(coro_fn, *args, max_retries=5, base_delay=15, **kwargs):
    """Retry a coroutine-returning function on RateLimitError with backoff."""
    for attempt in range(max_retries):
        try:
            return await coro_fn(*args, **kwargs)
        except RateLimitError as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (attempt + 1)
            print(f"⏳ Rate limited, retrying in {delay}s (attempt {attempt+1}/{max_retries})")
            await asyncio.sleep(delay)

In [ ]:
class DeepResearchWorkflow(Workflow):

    @step
    async def setup(self, ctx: Context, ev: StartEvent) -> GenerateEvent:
        self.llm = ev.llm
        self.answer_agent = ev.answer_agent
        # Limit how many answer_question steps run concurrently
        self.semaphore = asyncio.Semaphore(1)  # 1 = fully sequential; raise to 2-3 if you have more headroom

        ctx.write_event_to_stream(ProgressEvent(msg="Starting research"))
        return GenerateEvent(research_topic=ev.research_topic)

    @step
    async def generate_questions(self, ctx: Context, ev: GenerateEvent) -> QuestionEvent:
        await ctx.store.set("research_topic", ev.research_topic)
        ctx.write_event_to_stream(ProgressEvent(msg=f"Research topic is {ev.research_topic}"))

        prompt = f"""You are part of a deep research system.
          Given a research topic, you should come up with a bunch of questions
          that a separate agent will answer in order to write a comprehensive
          report on that topic. To make it easy to answer the questions separately,
          you should provide the questions one per line. Don't include markdown
          or any preamble in your response, just a list of questions.

          Topic: {ev.research_topic}"""

        result = await call_with_retry(self.llm.acomplete, prompt)

        lines = str(result).split("\n")
        questions = [line.strip() for line in lines if line.strip() != ""]

        # OPTIONAL but recommended: cap the number of questions to reduce total token usage
        questions = questions[:4]

        await ctx.store.set("total_questions", len(questions))

        for question in questions:
            ctx.send_event(QuestionEvent(question=question))

    @step
    async def answer_question(self, ctx: Context, ev: QuestionEvent) -> AnswerEvent:
        async with self.semaphore:  # only N of these run at once
            result = await call_with_retry(
                self.answer_agent.run,
                user_msg=f"""Research the answer to this
                  question: <question>{ev.question}</question>. You can use web
                  search to help you find information on the topic, as many times
                  as you need. Return just the answer without preamble or markdown.
                  Keep your answer concise (under 150 words)."""
            )
            # Small pause to spread out token usage across the minute
            await asyncio.sleep(5)

        ctx.write_event_to_stream(ProgressEvent(msg=f"""Received question {ev.question}
            Came up with answer: {str(result)}"""))

        return AnswerEvent(question=ev.question, answer=str(result))

    @step
    async def write_report(self, ctx: Context, ev: AnswerEvent) -> StopEvent:
        research = ctx.collect_events(ev, [AnswerEvent] * await ctx.store.get("total_questions"))
        if research is None:
            ctx.write_event_to_stream(ProgressEvent(msg="Collecting answers..."))
            return None

        ctx.write_event_to_stream(ProgressEvent(msg="Generating report..."))

        all_answers = ""
        for q_and_a in research:
            all_answers += f"Question: {q_and_a.question}\nAnswer: {q_and_a.answer}\n\n"

        prompt = f"""You are part of a deep research system.
          You have been given a complex topic on which to write a report:
          <topic>{await ctx.store.get("research_topic")}.

          Other agents have already come up with a list of questions about the
          topic and answers to those questions. Your job is to write a clear,
          thorough report that combines all the information from those answers.

          Here are the questions and answers:
          <questions_and_answers>{all_answers}</questions_and_answers>"""

        result = await call_with_retry(self.llm.acomplete, prompt)

        return StopEvent(result=str(result))

In [ ]:
workflow = DeepResearchWorkflow(timeout=None)

handler = workflow.run(
    research_topic="The history of the web, including the development of the "
                    "internet and the web, and 21st century developments",
    llm=llm,
    answer_agent=answer_agent,
)

async for event in handler.stream_events():
    if isinstance(event, ProgressEvent):
        print(f"📣 {event.msg}")

final_result = await handler

print("\n" + "="*50)
print("📄 FINAL REPORT")
print("="*50 + "\n")
print(final_result)

# **Adding Self-Reflection to the Workflow**

We now have a deep research agent! It comprehensively researches a topic for us before providing a detailed answer. But we can do better!

🧘 LLMs are capable of self-reflection: they can read their own work, critique it, and provide feedback, allowing them to take a second try when they fall short.

Let's add reflection to our deep research agent! This will involve several changes:

* In `research` we'll store the research into the context, since we might need to use it multiple times
* We'll tell `write` that it can be triggered by a `RewriteEvent` in addition to a `WriteEvent`
* If it's a `RewriteEvent` we'll add the review as feedback to the prompt
* `review` will be changed to optionally emit a `RewriteEvent`
* We'll get the LLM to decide if the review returned by the agent is a "bad" or "good" review

In [ ]:
from llama_index.core.workflow import (
    Event, Context, StartEvent, StopEvent, Workflow, step
)
from llama_index.core.agent.workflow import FunctionAgent

async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient()
    response = await client.search(
        query,
        max_results=3,          # fewer results
        search_depth="basic",   # avoid "advanced" which pulls more raw content
    )

    # Trim to just what's useful, and cap each result's content length
    trimmed = []
    for r in response.get("results", []):
        trimmed.append({
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": (r.get("content") or "")[:500],  # cap content length
        })

    return str({"results": trimmed})

async def record_notes(ctx: Context, notes: str, notes_title: str = "Untitled Notes") -> str:
    """Useful for recording notes on a given topic."""
    async with ctx.store.edit_state() as state:
        if "research_notes" not in state:
            state["research_notes"] = {}
        state["research_notes"][notes_title] = notes
    return "Notes recorded."

async def write_report(ctx: Context, report_content: str) -> str:
    """Useful for writing a report on a given topic."""
    async with ctx.store.edit_state() as state:
        state["report_content"] = report_content
    return "Report written."

async def review_report(ctx: Context, review: str) -> str:
    """Useful for reviewing a report and providing feedback."""
    async with ctx.store.edit_state() as state:
        state["review"] = review
    return "Report reviewed."


# --- Agents ---
research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Searches the web and records notes on a topic.",
    system_prompt=(
        "You are the ResearchAgent. Search the web for information on the "
        "given topic. Perform at most 2-3 searches total. After searching, "
        "write a CONCISE summary (under 300 words) of what you found using "
        "the record_notes tool, rather than including raw search results."
    ),
    llm=llm,
    tools=[search_web, record_notes],
)

write_agent = FunctionAgent(
    name="WriteAgent",
    description="Writes a report on a given topic.",
    system_prompt=(
        "You are the WriteAgent. Write a clear, thorough markdown report "
        "grounded in the provided research notes. If feedback from a "
        "previous review is provided, you MUST address every point in it. "
        "Save your report using the write_report tool."
    ),
    llm=llm,
    tools=[write_report],
)

review_agent = FunctionAgent(
    name="ReviewAgent",
    description="Reviews a report and gives feedback.",
    system_prompt=(
        "You are the ReviewAgent. Carefully assess the report for accuracy, "
        "thoroughness, and organization. Either approve it explicitly by "
        "starting your review with 'APPROVED', or list the specific changes "
        "the WriteAgent must make. Save your review using the review_report tool."
    ),
    llm=llm,
    tools=[review_report],
)


# --- Events ---
class WriteEvent(Event):
    pass

class RewriteEvent(Event):
    review: str

class ReviewEvent(Event):
    pass

class ProgressEvent(Event):
    msg: str


class ReflectiveResearchWorkflow(Workflow):

    MAX_REWRITES = 3

    @step
    async def research(self, ctx: Context, ev: StartEvent) -> WriteEvent:
        research_topic = ev.research_topic
        await ctx.store.set("research_topic", research_topic)
        await ctx.store.set("rewrite_count", 0)

        ctx.write_event_to_stream(ProgressEvent(msg=f"Researching: {research_topic}"))

        research_ctx = Context(research_agent)
        await research_agent.run(
            ctx=research_ctx,
            user_msg=f"""Research the topic: <topic>{research_topic}</topic>.
              Do at most 2-3 web searches, then record a concise summary
              (under 300 words) with the record_notes tool. Do not include
              raw search result text in your notes — synthesize it."""
        )

        research_state = await research_ctx.store.get_state()
        async with ctx.store.edit_state() as state:
            state["research_notes"] = research_state.get("research_notes", {})

        ctx.write_event_to_stream(ProgressEvent(msg="Research complete, notes recorded."))
        return WriteEvent()

    @step
    async def write(self, ctx: Context, ev: WriteEvent | RewriteEvent) -> ReviewEvent:
        state = await ctx.store.get_state()
        research_notes = state.get("research_notes", {})
        research_topic = state.get("research_topic", "")

        notes_str = ""
        for title, notes in research_notes.items():
            notes_str += f"### {title}\n{notes}\n\n"

        if isinstance(ev, RewriteEvent):
            ctx.write_event_to_stream(ProgressEvent(msg="Rewriting report with feedback..."))
            feedback_str = f"""
              Your previous draft was reviewed and requires changes.
              You MUST address this feedback:
              <feedback>{ev.review}</feedback>

              Here is your previous draft:
              <previous_report>{state.get("report_content", "")}</previous_report>
            """
        else:
            ctx.write_event_to_stream(ProgressEvent(msg="Writing initial report..."))
            feedback_str = ""

        # FIX: fresh Context for this agent call
        write_ctx = Context(write_agent)
        await write_agent.run(
            ctx=write_ctx,
            user_msg=f"""Write a comprehensive markdown report on the topic:
              <topic>{research_topic}</topic>

              Base it on these research notes:
              <research_notes>{notes_str}</research_notes>
              {feedback_str}

              Save the final report using the write_report tool."""
        )

        # FIX: merge report_content back into main workflow state
        write_state = await write_ctx.store.get_state()
        async with ctx.store.edit_state() as state:
            state["report_content"] = write_state.get("report_content", "")

        ctx.write_event_to_stream(ProgressEvent(msg="Report written."))
        return ReviewEvent()

    @step
    async def review(self, ctx: Context, ev: ReviewEvent) -> RewriteEvent | StopEvent:
        state = await ctx.store.get_state()
        report_content = state.get("report_content", "")

        ctx.write_event_to_stream(ProgressEvent(msg="Reviewing report..."))

        # FIX: fresh Context for this agent call
        review_ctx = Context(review_agent)
        await review_agent.run(
            ctx=review_ctx,
            user_msg=f"""Review this report. Either approve it or list the
              specific changes needed.

              <report>{report_content}</report>

              Save your review using the review_report tool."""
        )

        # FIX: pull the review back out and merge into main workflow state
        review_state = await review_ctx.store.get_state()
        review_text = review_state.get("review", "")
        async with ctx.store.edit_state() as state:
            state["review"] = review_text

        ctx.write_event_to_stream(ProgressEvent(msg=f"Review received: {review_text}"))

        verdict_prompt = f"""You are given a review of a report. Decide whether
          the review APPROVES the report as-is, or REQUESTS CHANGES.
          Respond with exactly one word: "good" if it approves the report,
          or "bad" if it requests changes.

          <review>{review_text}</review>"""

        verdict_result = await llm.acomplete(verdict_prompt)
        verdict = str(verdict_result).strip().lower()

        rewrite_count = state.get("rewrite_count", 0)

        if "bad" in verdict and rewrite_count < self.MAX_REWRITES:
            await ctx.store.set("rewrite_count", rewrite_count + 1)
            ctx.write_event_to_stream(
                ProgressEvent(msg=f"Review requests changes (attempt {rewrite_count + 1}). Rewriting...")
            )
            return RewriteEvent(review=review_text)

        ctx.write_event_to_stream(ProgressEvent(msg="Report approved (or max rewrites reached)."))
        return StopEvent(result=report_content)

In [ ]:
workflow = ReflectiveResearchWorkflow(timeout=None)

handler = workflow.run(
    research_topic="The history of the web, including the development of the "
                    "internet and the web, and 21st century developments",
)

async for event in handler.stream_events():
    if isinstance(event, ProgressEvent):
        print(f"📣 {event.msg}")

final_result = await handler

print("\n" + "="*50)
print("📄 FINAL REPORT")
print("="*50 + "\n")
print(final_result)